# 03 - Evaluate Predictions and Explanations

Inspect prediction scores, reason codes, and feature attributions.


In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
NOTEBOOKS_DIR = HERE if HERE.name == 'notebooks' else (HERE / 'notebooks')
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import text
from pipeline_bootstrap import bootstrap_environment, get_db_session

bootstrap_environment()


In [ ]:
PREDICTION_TYPE = 'risk_gnn'  # or 'corruption_risk'

db = get_db_session()
try:
    sql = (
        'SELECT id, entity_key, entity_type, score, reason_codes, details_json, window_end '
        'FROM ai_prediction '
        'WHERE prediction_type = :ptype '
        'ORDER BY window_end DESC, score DESC '
        'LIMIT 300'
    )
    rows = db.execute(text(sql), {'ptype': PREDICTION_TYPE}).fetchall()
finally:
    db.close()

df = pd.DataFrame(rows, columns=['id', 'entity_key', 'entity_type', 'score', 'reason_codes', 'details_json', 'window_end'])
df.head(20)


In [ ]:
if not df.empty:
    ax = df['score'].astype(float).plot(kind='hist', bins=25, title=f'{PREDICTION_TYPE} score distribution')
    ax.set_xlabel('score')
    plt.show()
else:
    print('No predictions found.')


In [ ]:
if not df.empty:
    top_ids = df['id'].head(5).tolist()
    db = get_db_session()
    try:
        sql = (
            'SELECT e.prediction_id, e.reason_codes, e.counterfactual_json, e.details_json '
            'FROM ai_explanation e '
            'WHERE e.prediction_id = ANY(:ids)'
        )
        rows = db.execute(text(sql), {'ids': top_ids}).fetchall()
    finally:
        db.close()

    for prediction_id, reasons, counterfactual, details in rows:
        print('prediction_id:', prediction_id)
        print('reason_codes:', reasons)
        print('counterfactual:', counterfactual)
        print('explanation_method:', (details or {}).get('explanation_method'))
        print('feature_attributions:', (details or {}).get('feature_attributions', [])[:3])
        print('---')
